<a href="https://colab.research.google.com/github/zia207/Causal_Inference_R/blob/main/Notebook/02_08_04_02_causal_timeseries_its_bsts_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1GkUjaigI4sLWYxQRJ8xK1q7Qk4Dzv3SL)

# 4.2 Interrupted Time Series and Bayesian Structural Time Series

When a well-defined intervention — a new law, a clinical protocol change, a public health campaign, an environmental regulation — is implemented at a known calendar date, the most natural causal question is: *did the time series change in a way it would not have changed otherwise?* Interrupted Time Series (ITS) analysis answers this question by comparing the observed post-intervention trajectory against a counterfactual extrapolated from the pre-intervention period. It is one of the most widely used quasi-experimental designs in public health, health policy, environmental regulation, and program evaluation precisely because it requires no control group — only a long pre-intervention series and a credible assumption that the pre-trend would have continued.

This chapter covers three interconnected designs. **Segmented regression ITS** fits separate linear trends before and after the breakpoint, estimating both a level change and a slope change at the intervention. **Bayesian Structural Time Series (BSTS)** using the `CausalImpact` package extends this by fitting a full state-space model to the pre-intervention period, extrapolating a probabilistic counterfactual, and computing the posterior distribution of the causal effect. **Comparative ITS (CITS)** strengthens validity by adding a parallel control series, removing concurrent secular trends that could otherwise be mistaken for intervention effects.

## Overview

### Segmented Regression ITS

The canonical ITS regression model for a single aggregate unit observed at times $t = 1, \ldots, T$, with intervention at $T^*$:

$$
Y_t = \beta_0 + \beta_1 t + \beta_2 D_t + \beta_3 (t - T^*) D_t + \varepsilon_t
$$

where:

- $\beta_0$: pre-intervention intercept (baseline level)
- $\beta_1$: pre-intervention slope (baseline trend)
- $\beta_2$: **level change** at the intervention — immediate step change
- $\beta_3$: **slope change** — difference in post-intervention vs. pre-intervention trend
- $D_t = \mathbf{1}[t \geq T^*]$: post-intervention indicator
- $(t - T^*) D_t$: time elapsed since intervention (0 before, $t - T^*$ after)

**Interpretation of parameters:**

| Parameter | Meaning | Causal Question |
|---|---|---|
| $\beta_2$ | Immediate level change at $T^*$ | Did the series jump at the intervention? |
| $\beta_3$ | Change in slope post-intervention | Did the trend accelerate or decelerate? |
| $\beta_2 + k\beta_3$ | Cumulative effect at $k$ periods post | What is the sustained effect at time $k$? |

**Critical assumption — Trend Continuity:** The pre-intervention trend would have continued unchanged in the absence of the intervention. Violations occur if: (a) a concurrent event coincides with $T^*$; (b) regression to the mean affects the post-period; (c) secular trends affect treated and control units differentially.

**Autocorrelation correction:** ITS residuals are almost always autocorrelated. OLS standard errors are invalid. Use:

- **Newey-West HAC standard errors** (`sandwich` package)
- **GLS with AR(1) correlation structure** (`nlme::gls` with `corAR1`)
- **ARIMA intervention models** (`forecast::Arima` with xreg)

### Bayesian Structural Time Series (BSTS / CausalImpact)

The `CausalImpact` package (Brodersen et al., 2015) models the pre-intervention time series as a **state-space model**:

$$
y_t = \mathbf{Z}_t^\top \boldsymbol{\alpha}_t + \varepsilon_t, \quad \varepsilon_t \sim \mathcal{N}(0, \sigma_\varepsilon^2)
$$

$$
\boldsymbol{\alpha}_{t+1} = \mathbf{T}_t \boldsymbol{\alpha}_t + \mathbf{R}_t \boldsymbol{\eta}_t, \quad \boldsymbol{\eta}_t \sim \mathcal{N}(\mathbf{0}, \mathbf{Q}_t)
$$

The state vector $\boldsymbol{\alpha}_t$ captures a **local linear trend** (level + slope), **seasonal components**, and optionally **covariate regression** components. The model is fitted to the pre-intervention period only; the posterior predictive distribution is then extrapolated forward as the counterfactual.

The causal effect at time $t$ (post-intervention) is:

$$
\delta_t = y_t - \hat{y}_t^{\text{counterfactual}}
$$

The package returns point-wise effects, cumulative effects, and posterior credible intervals — all as probability distributions, not single-point estimates.

**Advantages over segmented regression:**

| Segmented Regression | BSTS / CausalImpact |
|---|---|
| Linear counterfactual only | Flexible non-linear state-space counterfactual |
| Point estimate with SE | Full posterior distribution of effect |
| Manual autocorrelation correction | Handles autocorrelation automatically |
| No covariate incorporation | Covariate regression built in |
| No uncertainty about counterfactual model | Model uncertainty propagated to effect estimate |

### Comparative ITS (CITS)

A CITS adds a control series $C_t$ to the ITS model. The control series experiences the same secular trends and concurrent events as the treated series, but is *not* directly exposed to the intervention. The CITS estimator is a **difference-in-discontinuities**:

$$
\hat{\delta}^{\text{CITS}} = \underbrace{(\text{change in } Y \text{ at } T^*)}_{\text{treated discontinuity}} - \underbrace{(\text{change in } C \text{ at } T^*)}_{\text{control discontinuity}}
$$

This removes the bias from any concurrent event that affects both series equally at $T^*$.

## Implementation in R

This section provides a complete production-ready workflow: simulated data for pedagogical clarity, followed by real-world ITS, BSTS, and CITS applications.

## Setup R in Python Runtime - Install {rpy2}

{rpy2} allows running R code in Colab Python runtime via `%%R` magic.

In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython

## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Check and Install Required R Packages

In [ ]:
%%R
packages <- c(
  "tidyverse",
  "CausalImpact",
  "nlme",
  "sandwich",
  "lmtest",
  "forecast",
  "ggplot2",
  "patchwork"
)

In [ ]:
%%R
# Install missing packages
new_packages <- packages[!(packages %in% installed.packages(lib = 'drive/My Drive/R/')[,"Package"])]
if (length(new_packages)) install.packages(new_packages, lib = 'drive/My Drive/R/')

### Verify Installation

In [ ]:
%%R
# set library path
.libPaths('drive/My Drive/R')
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))

### Load Packages

In [ ]:
%%R
invisible(lapply(packages, function(pkg) {
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}))

### Check Loaded Packages

In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])

In [ ]:
%%R
set.seed(2026)
options(scipen = 999)

## Simulated Data: Segmented Regression ITS

### Data Generation

We simulate a monthly health outcome time series (e.g., hospitalisation rate per 100,000) with a known level change and slope change at an intervention implemented at month 37.

In [ ]:
%%R
# Parameters
T_total <- 72        # 72 months (6 years)
T_star  <- 37        # Intervention at month 37

# True effects
level_change <- -8    # Immediate drop of 8 units
slope_change <- -0.3  # Additional downward slope post-intervention

# Generate the series
its_df <- tibble(
  t        = 1:T_total,
  D        = as.integer(t >= T_star),          # Post-intervention indicator
  t_post   = pmax(t - T_star, 0) * D,          # Time since intervention
  # True data-generating process
  Y_true   = 80 + 0.4 * t                       # Pre-trend
             + level_change * D
             + slope_change * t_post,
  # Observed with AR(1) autocorrelated noise
  Y        = Y_true + arima.sim(n = T_total,
               model = list(ar = 0.55, sd = 3))
)

cat("ITS Dataset Summary:\n")
cat(sprintf("  Observations: %d (pre: %d, post: %d)\n",
            T_total, T_star - 1, T_total - T_star + 1))
cat(sprintf("  True level change: %.1f\n  True slope change: %.2f\n",
            level_change, slope_change))

### Visualizing the ITS Design

In [ ]:
%%R -w 1000 -h 500
ggplot(its_df, aes(x = t, y = Y)) +
  geom_point(alpha = 0.5, size = 1.5, color = "gray40") +
  geom_line(alpha = 0.4, color = "gray40") +
  geom_vline(xintercept = T_star - 0.5, linetype = "dashed",
             color = "red", linewidth = 1.1) +
  # Pre-intervention LM
  geom_smooth(data = filter(its_df, D == 0),
              method = "lm", se = TRUE,
              color = "steelblue", linewidth = 1.3, fill = "lightblue") +
  # Post-intervention LM
  geom_smooth(data = filter(its_df, D == 1),
              method = "lm", se = TRUE,
              color = "firebrick", linewidth = 1.3, fill = "mistyrose") +
  annotate("text", x = T_star + 1, y = max(its_df$Y) * 0.97,
           label = "Intervention\nat t=37", hjust = 0, color = "red", size = 4) +
  labs(
    title    = "Interrupted Time Series: Simulated Hospitalisation Rate",
    subtitle = "Blue = pre-intervention trend; Red = post-intervention trend",
    x = "Month", y = "Rate per 100,000"
  ) +
  theme_minimal(base_size = 14)

### Segmented Regression Estimation

In [ ]:
%%R
# Fit standard OLS ITS model
its_ols <- lm(Y ~ t + D + t_post, data = its_df)
summary(its_ols)

### Autocorrelation Diagnosis

In [ ]:
%%R -w 800 -h 400
# Durbin-Watson test for autocorrelation
dw <- dwtest(its_ols)
cat(sprintf("Durbin-Watson statistic: %.4f (p = %.4f)\n",
            dw$statistic, dw$p.value))
cat(ifelse(dw$p.value < 0.05,
           "\u2192 Significant autocorrelation \u2014 correct standard errors!\n",
           "\u2192 No significant autocorrelation detected.\n"))

# ACF of residuals
acf(residuals(its_ols), main = "ACF of OLS Residuals")

### Newey-West HAC Standard Errors

In [ ]:
%%R
# Corrected inference with Newey-West HAC SEs
its_nw <- coeftest(its_ols, vcov = NeweyWest(its_ols, lag = 4))

cat("=== ITS Estimates with Newey-West SEs ===\n")
print(its_nw)
cat(sprintf("\nLevel change  (beta_2): %.3f (SE: %.3f, p = %.4f)\n",
            its_nw["D", "Estimate"],
            its_nw["D", "Std. Error"],
            its_nw["D", "Pr(>|t|)"]))
cat(sprintf("Slope change  (beta_3): %.3f (SE: %.3f, p = %.4f)\n",
            its_nw["t_post", "Estimate"],
            its_nw["t_post", "Std. Error"],
            its_nw["t_post", "Pr(>|t|)"]))
cat(sprintf("\nTrue level change: %.1f | True slope change: %.2f\n",
            level_change, slope_change))

### GLS with AR(1) Correlation (Preferred Approach)

In [ ]:
%%R
# GLS with AR(1) correlation structure — more efficient than NW
its_gls <- gls(Y ~ t + D + t_post,
               data      = its_df,
               correlation = corAR1(form = ~ 1))
summary(its_gls)

### Predicted vs. Counterfactual Plot

In [ ]:
%%R -w 1000 -h 500
# Generate counterfactual (no intervention: D=0, t_post=0 for all periods)
cf_df <- its_df %>%
  mutate(
    Y_fitted       = fitted(its_gls),
    Y_counterfactual = predict(its_gls,
                                newdata = its_df %>% mutate(D = 0, t_post = 0))
  )

ggplot(cf_df, aes(x = t)) +
  geom_point(aes(y = Y), alpha = 0.4, size = 1.4, color = "gray40") +
  geom_line(aes(y = Y_fitted), color = "steelblue", linewidth = 1.2, linetype = "solid") +
  geom_line(aes(y = Y_counterfactual), color = "firebrick", linewidth = 1.2, linetype = "dashed") +
  geom_vline(xintercept = T_star - 0.5, linetype = "dotted", color = "red", linewidth = 1) +
  annotate("text", x = T_star + 1, y = max(cf_df$Y) * 0.96,
           label = "Intervention", hjust = 0, color = "red", size = 4) +
  labs(
    title    = "ITS: Observed vs. Counterfactual (No Intervention)",
    subtitle = "Solid = GLS fitted values | Dashed = counterfactual trend",
    x = "Month", y = "Rate per 100,000"
  ) +
  theme_minimal(base_size = 14)

## Bayesian Structural Time Series: CausalImpact

### CausalImpact on Simulated Data

In [ ]:
%%R
# Prepare time series object for CausalImpact
# CausalImpact requires a zoo object, data frame, or numeric vector — not ts()
y_ts <- zoo(its_df$Y)

# Define pre- and post-intervention periods
pre_period  <- c(1, T_star - 1)
post_period <- c(T_star, T_total)

# Run CausalImpact
ci_result <- CausalImpact(
  data       = y_ts,
  pre.period = pre_period,
  post.period = post_period,
  model.args = list(
    niter              = 2000,  # MCMC iterations
    standardize.data   = TRUE,
    prior.level.sd     = 0.01,
    nseasons           = 1
  )
)

### CausalImpact Summary

In [ ]:
%%R
summary(ci_result)

### CausalImpact Plots

In [ ]:
%%R -w 1000 -h 800
plot(ci_result,
     c("original", "pointwise", "cumulative"))

### Extracting Posterior Estimates

In [ ]:
%%R
# Extract summary statistics
ci_summary <- summary(ci_result)

cat("=== CausalImpact Posterior Summary ===\n")
cat(sprintf("Average causal effect:      %.3f (95%% CI: %.3f, %.3f)\n",
            ci_summary$Average[1, "Actual"] - ci_summary$Average[1, "Pred"],
            ci_summary$Average[1, "Pred.lower"],
            ci_summary$Average[1, "Pred.upper"]))
cat(sprintf("Cumulative causal effect:   %.3f\n",
            ci_summary$Cumulative[1, "AbsEffect"]))
cat(sprintf("Posterior probability:      %.3f\n",
            1 - ci_summary$Average[1, "p"]))

## CausalImpact with Control Covariates

We demonstrate CausalImpact using the synthetic example from the official package vignette (Brodersen et al., 2015): 200 time points, an intervention at $t = 71$, two control covariates ($x_1$, $x_2$) that share pre-intervention dynamics with the outcome but are unaffected by the intervention.

### Generate Example Data

In [ ]:
%%R
set.seed(2026)
n_real    <- 200
interv_t  <- 71

# Two control covariates following correlated random walks
x1 <- 100 + arima.sim(model = list(ar = 0.999), n = n_real, sd = 1)
x2 <-  80 + arima.sim(model = list(ar = 0.95),  n = n_real, sd = 1.5)

# Outcome driven by x1 pre-intervention; +10 causal lift post-intervention
y <- 1.2 * x1 + 0.5 * x2 + rnorm(n_real, sd = 3)
y[interv_t:n_real] <- y[interv_t:n_real] + 10

# Combine into a zoo object (required by CausalImpact)
impact_data <- zoo(cbind(y, x1, x2))

cat("Dataset dimensions:", dim(impact_data), "\n")
cat("Column names:", colnames(impact_data), "\n")
cat(sprintf("Pre-period: 1-%d | Post-period: %d-%d\n\n",
            interv_t - 1, interv_t, n_real))

# Quick plot of outcome
df_real <- as_tibble(as.data.frame(impact_data)) %>%
  mutate(t = 1:n(),
         phase = ifelse(t < interv_t, "Pre", "Post"))

In [ ]:
%%R -w 1000 -h 450
ggplot(df_real, aes(x = t, y = y, color = phase)) +
  geom_line(linewidth = 0.9, aes(group = 1)) +
  geom_vline(xintercept = interv_t - 0.5, linetype = "dashed", color = "red") +
  scale_color_manual(values = c("Pre" = "steelblue", "Post" = "firebrick")) +
  labs(title = "CausalImpact Example: Outcome Series with +10 Intervention Effect",
       subtitle = "Vertical line = intervention at t=71",
       x = "Time", y = "Outcome Y") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "bottom")

### CausalImpact with Control Covariates

In [ ]:
%%R -w 1000 -h 800
pre_real  <- c(1, interv_t - 1)
post_real <- c(interv_t, n_real)

ci_real <- CausalImpact(
  data        = impact_data,      # zoo object: y, x1, x2
  pre.period  = pre_real,
  post.period = post_real,
  model.args  = list(
    niter            = 3000,
    standardize.data = TRUE,
    prior.level.sd   = 0.01
  )
)

summary(ci_real)
plot(ci_real)

## Comparative ITS (CITS)

### Simulated CITS with Concurrent Confounder

We simulate a concurrent secular trend that affects both the treated series $Y$ and a control series $C$ at $T^*$, plus a true intervention effect on $Y$ only. The naive ITS overestimates the effect; CITS removes the bias.

In [ ]:
%%R
T3     <- 72
T_star3 <- 37

# Concurrent confounding at T_star (e.g., a seasonal shock)
concurrent_shock <- 5   # affects both Y and C
true_effect      <- -8  # true intervention effect on Y only

cits_df <- tibble(
  t      = 1:T3,
  D      = as.integer(t >= T_star3),
  t_post = pmax(t - T_star3, 0) * D,
  # Treated series: true effect + concurrent shock
  Y_true  = 80 + 0.3 * t + (true_effect + concurrent_shock) * D - 0.2 * t_post,
  Y       = Y_true + rnorm(T3, sd = 2.5),
  # Control series: only concurrent shock (no true effect)
  C_true  = 75 + 0.3 * t + concurrent_shock * D,
  C       = C_true + rnorm(T3, sd = 2.5)
)

cat("True intervention effect on Y:", true_effect, "\n")
cat("Concurrent shock (affects both):", concurrent_shock, "\n")
cat("Naive ITS would estimate:", true_effect + concurrent_shock, "(biased)\n")
cat("CITS removes concurrent shock to recover:", true_effect, "\n")

### Naive ITS on Treated Series

In [ ]:
%%R
# Naive ITS: ignores concurrent shock
its_naive <- coeftest(
  lm(Y ~ t + D + t_post, data = cits_df),
  vcov = NeweyWest(lm(Y ~ t + D + t_post, data = cits_df), lag = 3)
)
cat("=== Naive ITS Level Change ===\n")
cat(sprintf("Estimate: %.3f (bias: %.3f)\n",
            its_naive["D", "Estimate"],
            its_naive["D", "Estimate"] - true_effect))

### CITS: Difference-in-Discontinuities

In [ ]:
%%R
# Stack treated and control into long format
cits_long <- bind_rows(
  cits_df %>% mutate(series = "treated", outcome = Y),
  cits_df %>% mutate(series = "control", outcome = C)
) %>%
  mutate(treated_series = as.integer(series == "treated"))

# CITS model: include series interaction
cits_model <- lm(
  outcome ~ t + D + t_post + treated_series +
    treated_series:D + treated_series:t_post,
  data = cits_long
)
cits_nw <- coeftest(cits_model, vcov = NeweyWest(cits_model, lag = 3))

cat("=== CITS Model: Difference-in-Discontinuities ===\n")
print(cits_nw)

# R may order the interaction as "D:treated_series" or "treated_series:D"
# depending on where each main effect first appears in the formula
ts_D_nm <- grep("treated_series.*D|D.*treated_series", rownames(cits_nw), value = TRUE)[1]
cat(sprintf("\nCITS level change (%s): %.3f\n",
            ts_D_nm, cits_nw[ts_D_nm, "Estimate"]))
cat(sprintf("True intervention effect: %.1f\n", true_effect))
cat(sprintf("Bias removed: %.3f (vs. naive bias %.3f)\n",
            abs(cits_nw[ts_D_nm, "Estimate"] - true_effect),
            abs(its_naive["D", "Estimate"] - true_effect)))

### CITS Visualization

In [ ]:
%%R -w 1000 -h 500
cits_plot <- cits_df %>%
  pivot_longer(cols = c(Y, C), names_to = "Series", values_to = "Value")

ggplot(cits_plot, aes(x = t, y = Value, color = Series)) +
  geom_point(alpha = 0.4, size = 1.2) +
  geom_smooth(data = filter(cits_plot, D == 0),
              method = "lm", se = FALSE, linewidth = 1.2) +
  geom_smooth(data = filter(cits_plot, D == 1),
              method = "lm", se = FALSE, linewidth = 1.2, linetype = "dashed") +
  geom_vline(xintercept = T_star3 - 0.5, linetype = "dotted",
             color = "red", linewidth = 1.1) +
  scale_color_manual(values = c("Y" = "steelblue", "C" = "darkorange")) +
  labs(
    title    = "Comparative ITS (CITS): Treated vs. Control Series",
    subtitle = "Solid = pre-trend | Dashed = post-trend | Control removes concurrent confounding",
    x = "Month", y = "Outcome"
  ) +
  theme_minimal(base_size = 14) +
  theme(legend.position = "bottom")

## Advanced Diagnostics

### Sensitivity: Alternative Counterfactual Models

A key diagnostic for ITS is to test whether results are sensitive to the functional form of the counterfactual. We compare linear, quadratic, and ARIMA-based counterfactuals.

In [ ]:
%%R
# Linear counterfactual (already fitted above as its_gls)
cf_linear <- predict(its_gls,
                      newdata = its_df %>% mutate(D = 0, t_post = 0))

# Quadratic counterfactual
its_quad <- lm(Y ~ t + I(t^2) + D + t_post, data = its_df)
cf_quad <- predict(its_quad,
                   newdata = its_df %>% mutate(D = 0, t_post = 0))

# Effect estimates at last post-period
eff_linear <- its_df$Y[T_total] - cf_linear[T_total]
eff_quad   <- its_df$Y[T_total] - cf_quad[T_total]

cat("=== Sensitivity to Counterfactual Model Form ===\n")
cat(sprintf("Linear counterfactual \u2014 effect at t=%d: %.3f\n", T_total, eff_linear))
cat(sprintf("Quadratic counterfactual \u2014 effect at t=%d: %.3f\n", T_total, eff_quad))
cat(sprintf("True effect at t=%d: %.1f\n", T_total,
            level_change + slope_change * (T_total - T_star)))

### Placebo Test: Pre-Intervention False Breakpoint

In [ ]:
%%R
# Placebo test: set fake intervention at t=20 (pre-intervention period)
placebo_df <- its_df %>%
  filter(t <= T_star - 1) %>%
  mutate(
    t_placebo    = t,
    D_placebo    = as.integer(t >= 20),
    t_post_placebo = pmax(t - 20, 0) * D_placebo
  )

placebo_model <- lm(Y ~ t_placebo + D_placebo + t_post_placebo, data = placebo_df)
placebo_nw <- coeftest(placebo_model, vcov = NeweyWest(placebo_model, lag = 3))

cat("=== Placebo Test (False Intervention at t=20) ===\n")
cat(sprintf("Placebo level change: %.3f (p = %.4f)\n",
            placebo_nw["D_placebo", "Estimate"],
            placebo_nw["D_placebo", "Pr(>|t|)"]))
cat(ifelse(placebo_nw["D_placebo", "Pr(>|t|)"] > 0.05,
           "\u2192 Non-significant placebo \u2014 no spurious detection (good)\n",
           "\u2192 Significant placebo \u2014 pre-trend concerns!\n"))

## Summary and Conclusion

This chapter equipped you with the full ITS toolkit from simple segmented regression to Bayesian counterfactual modeling and comparative designs. Key takeaways:

**On segmented regression.** The two-parameter ITS model ($\beta_2$ = level change, $\beta_3$ = slope change) is simple and interpretable. However, OLS standard errors are almost always invalid due to autocorrelation — always apply Newey-West HAC or GLS-AR(1) corrections. The choice of correction matters and should be reported transparently.

**On CausalImpact.** The BSTS approach provides a richer counterfactual than linear segmented regression: it incorporates trend, seasonality, and covariate information, and propagates model uncertainty to the effect estimate. Control covariates (series that share common trends with the treated series but are unaffected by the intervention) substantially improve counterfactual accuracy.

**On CITS.** Whenever a plausible control series exists, use it. The difference-in-discontinuities design removes concurrent confounding that would bias simple ITS estimates. Even an imperfect control series that partially shares common trends improves validity over no control at all.

**On diagnostics.** Every ITS should report: (a) a placebo test at a false breakpoint in the pre-period; (b) sensitivity to alternative counterfactual specifications; (c) a search for concurrent events at $T^*$ (the most common real-world threat to ITS validity); and (d) autocorrelation diagnostics with corrected standard errors.

## Resources

| Resource | Description |
|---|---|
| **Brodersen et al. (2015)** | *Annals of Applied Statistics* — CausalImpact paper |
| **`CausalImpact` package** | Google GitHub — vignette and worked examples |
| **Bernal et al. (2017)** | *International J. Epidemiology* — ITS tutorial for public health |
| **Penfold & Zhang (2013)** | *Psychiatr Serv* — practical ITS guide |
| **Wagner et al. (2002)** | *J. Clinical Pharmacy* — segmented regression review |
| **`nlme` package** | Pinheiro & Bates — GLS with correlated errors |
| **`sandwich` package** | Zeileis — HAC standard errors |